# 13 — Batch Processing & Evaluation Pipelines

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anulum/director-ai/blob/main/notebooks/13_batch_processing_and_evaluation.ipynb)

Score hundreds or thousands of LLM outputs in parallel.
Build evaluation pipelines for benchmarking, regression testing,
and dataset quality assessment.

**What you'll learn:**
1. `BatchProcessor` for parallel scoring
2. Async batch processing
3. Building an evaluation pipeline
4. Claim-level attribution
5. Conversation session tracking

In [ ]:
!pip install -q director-ai

## 1. Batch Processing

In [ ]:
from director_ai.core import CoherenceScorer, GroundTruthStore

# Build knowledge base
store = GroundTruthStore()
store.add("capital_france", "The capital of France is Paris.")
store.add("capital_germany", "The capital of Germany is Berlin.")
store.add("boiling_point", "Water boils at 100 degrees C at sea level.")
store.add("speed_of_light", "The speed of light is approximately 3e8 m/s.")

scorer = CoherenceScorer(threshold=0.3, ground_truth_store=store, use_nli=False)

# Batch of (prompt, response) pairs to evaluate
items = [
    ("Capital of France?", "The capital is Paris."),
    ("Capital of Germany?", "The capital is Berlin."),
    ("Boiling point of water?", "Water boils at 100 degrees C."),
    ("Speed of light?", "Light travels at about 300 km/s."),  # wrong
    ("Capital of France?", "The capital is Tokyo."),  # wrong
]

results = scorer.review_batch(items)

for (prompt, response), (approved, score) in zip(items, results):
    status = "PASS" if approved else "FAIL"
    print(f"  [{status}] {score.score:.3f}  {response[:60]}")

## 2. Evaluation Pipeline

Build a reusable evaluation pipeline for regression testing.

In [ ]:
def evaluate_dataset(scorer, dataset: list[dict]) -> dict:
    """Evaluate a dataset of {prompt, response, expected_label} entries.

    Returns accuracy, precision, recall, F1, and per-item details.
    """
    tp = fp = tn = fn = 0
    details = []

    for entry in dataset:
        approved, score = scorer.review(entry["prompt"], entry["response"])
        expected = entry["expected_label"]  # True=should pass, False=should fail

        if approved and expected:
            tp += 1
        elif approved and not expected:
            fp += 1
        elif not approved and expected:
            fn += 1
        else:
            tn += 1

        details.append(
            {
                "prompt": entry["prompt"][:50],
                "approved": approved,
                "expected": expected,
                "correct": approved == expected,
                "score": score.score,
            },
        )

    total = tp + fp + tn + fn
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (
        2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    )

    return {
        "accuracy": (tp + tn) / total if total > 0 else 0,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion": {"tp": tp, "fp": fp, "tn": tn, "fn": fn},
        "details": details,
    }


# Synthetic evaluation dataset
eval_data = [
    {
        "prompt": "Capital of France?",
        "response": "Paris is the capital.",
        "expected_label": True,
    },
    {
        "prompt": "Capital of France?",
        "response": "London is the capital.",
        "expected_label": False,
    },
    {
        "prompt": "Boiling point?",
        "response": "Water boils at 100°C.",
        "expected_label": True,
    },
    {
        "prompt": "Boiling point?",
        "response": "Water boils at 50°C.",
        "expected_label": False,
    },
    {
        "prompt": "Speed of light?",
        "response": "About 300,000 km/s.",
        "expected_label": True,
    },
    {
        "prompt": "Speed of light?",
        "response": "About 100 km/h.",
        "expected_label": False,
    },
]

results = evaluate_dataset(scorer, eval_data)

print(f"Accuracy:  {results['accuracy']:.1%}")
print(f"Precision: {results['precision']:.1%}")
print(f"Recall:    {results['recall']:.1%}")
print(f"F1:        {results['f1']:.1%}")
print(f"\nConfusion: {results['confusion']}")

print("\nDetails:")
for d in results["details"]:
    mark = "✓" if d["correct"] else "✗"
    print(f"  {mark} [{d['score']:.3f}] {d['prompt']}")

## 3. Claim-Level Attribution

The scorer can decompose a response into individual claims
and attribute each to a ground truth source.

In [ ]:
from director_ai.core import CoherenceScorer, GroundTruthStore

store = GroundTruthStore()
store.add("france", "Paris is the capital of France. Population: 2.1 million.")
store.add("tower", "The Eiffel Tower is 330 metres tall. Built in 1889.")

scorer = CoherenceScorer(
    threshold=0.5,
    ground_truth_store=store,
    use_nli=False,
)

# Multi-claim response
response = (
    "Paris is the capital of France. "
    "The Eiffel Tower is 330 metres tall. "
    "It was built in 1850."
)

approved, score = scorer.review("Tell me about Paris", response)

print(f"Overall: approved={approved}, score={score.score:.3f}")

if score.evidence and score.evidence.attributions:
    print("\nClaim attributions:")
    for attr in score.evidence and score.evidence.attributions:
        print(f"  [{(1 - attr.divergence):.3f}] {attr.claim}")
        if attr.source_sentence:
            print(f"         ↳ matched: {attr.source_sentence}")

## 4. Conversation Session Tracking

Track coherence across a multi-turn conversation.

In [ ]:
from director_ai.core import CoherenceScorer, ConversationSession, GroundTruthStore

store = GroundTruthStore()
store.add("product", "Widget Pro costs 49 USD. Ships in 2 business days.")
store.add("warranty", "1-year warranty. No refunds after 30 days.")

scorer = CoherenceScorer(threshold=0.3, ground_truth_store=store, use_nli=False)
session = ConversationSession(max_turns=10)

conversations = [
    ("How much is Widget Pro?", "Widget Pro is 49 USD with free shipping."),
    ("What about the warranty?", "It comes with a 1-year warranty."),
    ("Can I return it after 6 months?", "Yes, full refund any time."),  # hallucination
]

for prompt, response in conversations:
    approved, cs = scorer.review(prompt, response)
    turn = session.add_turn(prompt, response, cs.score)
    status = "PASS" if approved else "FAIL"
    print(f"Turn {turn.turn_index}: [{status}] score={cs.score:.3f}")

print(f"
Session: {len(session.turns)} turns")
avg = sum(t.score for t in session.turns) / len(session.turns)
print(f"Average coherence: {avg:.3f}")

## 5. Regression Testing Pattern

Integrate evaluation into CI to catch regressions.

In [ ]:
def regression_gate(scorer, test_cases: list[dict], min_accuracy: float = 0.9) -> bool:
    """CI gate: fails if accuracy drops below threshold."""
    results = evaluate_dataset(scorer, test_cases)
    accuracy = results["accuracy"]

    print(f"Regression gate: accuracy={accuracy:.1%} (min={min_accuracy:.1%})")

    if accuracy < min_accuracy:
        print("FAILED — accuracy below threshold")
        # Print failing cases
        for d in results["details"]:
            if not d["correct"]:
                print(f"  ✗ {d['prompt']} → score={d['score']:.3f}")
        return False

    print("PASSED")
    return True


# Run gate
passed = regression_gate(scorer, eval_data, min_accuracy=0.8)
print(f"\nGate result: {'PASS' if passed else 'FAIL'}")

## Summary

| Feature | Class | Use Case |
|---------|-------|----------|
| Batch scoring | `CoherenceScorer.review_batch()` | Bulk evaluation, dataset QA |
| Evaluation pipeline | Custom | Accuracy/F1 measurement |
| Claim attribution | `CoherenceScorer` | Per-claim fact checking |
| Session tracking | `ConversationSession` | Multi-turn coherence |
| Regression gate | Custom | CI integration |
